# Lesson 1.2 — Designing the End-to-End Architecture

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Model a Claude solution as four stages: input -> processing -> output -> feedback
- Identify where validation, retrieval, and human review attach to each stage
- Express the architecture as inspectable, reviewable code (diagram-as-data)

**You will produce:** An `architecture_spec.json` describing stages, components, and the feedback loop.

## Concepts

Every production Claude system is a **pipeline with a feedback loop**, not a single API call. The four canonical stages:

1. **Input** — ingestion, normalization, PII handling, request validation.
2. **Processing** — prompt assembly, retrieval (RAG), the model call(s), tool use.
3. **Output** — parsing/validation, guardrails, formatting, delivery.
4. **Feedback** — logging, evaluation, human review, data that improves the next version.

**RAG** (Retrieval-Augmented Generation) means fetching relevant documents and placing them in the prompt so the model answers from your data — we design it fully in Phase 3.

The feedback loop is what separates an architect's design from a demo: it is how the system earns trust and improves. Omitting it is a classic exam-level mistake.

> **Reference:** [Anthropic — Agents and workflows](https://docs.anthropic.com/en/docs/build-with-claude/building-effective-agents)

In [ ]:
# !pip install anthropic
# Guarded Claude client setup — every lesson runs with or without an API key.
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic()
        LIVE = True
    else:
        client = None
        LIVE = False
except Exception as e:  # SDK not installed in this kernel
    client = None
    LIVE = False
    print("anthropic SDK unavailable:", e)

# Current Claude model family (confirm exact IDs at https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = {
    "opus": "claude-opus-4-20250514",    # deepest reasoning, highest cost/latency
    "sonnet": "claude-sonnet-4-20250514",# balanced default workhorse
    "haiku": "claude-3-5-haiku-20241022",# fastest/cheapest for simple tasks
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    """Live call when a key exists; otherwise a deterministic mock so notebooks run."""
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system or "You are a helpful solution architect.",
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Model the four stages as components

We describe each stage as a list of components with the risk each one controls. Keeping this as data lets us validate and diff designs.

In [ ]:
architecture = {
  "input": [
    {"component": "fax/PDF ingest", "controls": "format normalization"},
    {"component": "PHI redaction gate", "controls": "HIPAA exposure"},
  ],
  "processing": [
    {"component": "policy RAG retriever", "controls": "grounding/hallucination"},
    {"component": "Claude Sonnet reasoner", "controls": "recommendation quality"},
  ],
  "output": [
    {"component": "schema validator", "controls": "malformed decisions"},
    {"component": "citation check", "controls": "unsupported claims"},
  ],
  "feedback": [
    {"component": "nurse review UI", "controls": "human-in-the-loop"},
    {"component": "decision log + eval set", "controls": "drift detection"},
  ],
}
for stage, comps in architecture.items():
    print(stage.upper())
    for c in comps:
        print(f"  - {c['component']}  ->  controls {c['controls']}")

## Code-Along: Step 2 — Validate the design has a closed feedback loop

An architecture review should fail fast if the feedback stage is empty or if no human-in-the-loop exists for a high-stakes decision.

In [ ]:
def review_architecture(arch, high_stakes=True):
    issues = []
    if not arch.get("feedback"):
        issues.append("No feedback loop: system cannot improve or detect drift.")
    comps = [c['component'] for s in arch.values() for c in s]
    if high_stakes and not any("review" in c.lower() for c in comps):
        issues.append("High-stakes but no human review component.")
    if not any("valid" in c.lower() or "schema" in c.lower() for c in comps):
        issues.append("No output validation — downstream systems may break.")
    return issues or ["PASS: closed loop with review and validation"]

for line in review_architecture(architecture):
    print(line)

## Code-Along: Step 3 — Persist the architecture spec

Write the spec to JSON so it can be versioned, reviewed in a PR, and consumed by the capstone.

In [ ]:
import json
spec = {"name": "prior-auth-triage", "stages": architecture,
        "review": review_architecture(architecture)}
with open("architecture_spec.json", "w", encoding="utf-8") as fh:
    json.dump(spec, fh, indent=2)
print(json.dumps(spec, indent=2)[:500])

## Exercise

Extend the architecture with an **escalation path**: when the citation check fails or the model is low-confidence, route to a senior nurse instead of auto-drafting. Add the component(s) and make `review_architecture` require an escalation path for high-stakes flows.

In [ ]:
# TODO: add an 'escalation' concept to the architecture dict
# TODO: extend review_architecture to require an escalation component when high_stakes=True
# TODO: re-run the review and persist the updated spec

## Key Takeaways

- Think in four stages: input, processing, output, feedback.
- Attach each component to the specific risk it controls.
- A design without a feedback loop is a demo, not an architecture.
- Diagram-as-data makes designs reviewable, diffable, and testable.

---

**Next:** [Lesson 1.3 — Selecting the Architectural Pattern](./lesson-1.3-pattern-selection.ipynb)